<a href="https://colab.research.google.com/github/zoesuhnny/data_science_and_ml_notes/blob/main/poem_lora_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
!pip install transformers peft "torchao>=0.16.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

In [13]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_id = "zoesunny/poem-lora-final"

#load base model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id,
                                             dtype=torch.bfloat16,
                                             device_map="auto",)

#apply LoRA adapter on top of base model
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval() #eval mode

#run test generation
messages = [
    {"role": "user",
     "content": "Analyze the haiku and classify it's sentiment and explain: Cold tea on the desk,/Unread words upon the screen,/Winter comes too soon."
}
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",  return_dict=True).to(model.device)

with torch.no_grad():
  output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature = 0.85)

print(tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The haiku is in English, with "Cold" being a negative sentiment, as it implies a cold winter atmosphere. The sentiment of the poem is generally negative, as the speaker is expressing their frustration or dissatisfaction with the upcoming winter season.
